# Pretraining-only Wavelet evaluation

This notebook **does not fine-tune or update weights**. It loads the completed 80k pretraining checkpoint and visualizes the Wavelet-registration pipeline for one fixed different-patient pair.


In [1]:
import os
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential
import torch.optim as optim
import voxelmorph as vxm
import neurite as ne
import scipy.ndimage

os.environ['VXM_BACKEND'] = 'pytorch'

backend:pytorch
Pytorch


In [2]:
os.environ.get('VXM_BACKEND')

'pytorch'

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [4]:
# 画像を読み込み
x_train = np.load('Data/TrainData_NoBed.npz')['Train']
x_train = np.transpose(x_train, (3, 0, 1, 2))

print('Resized train vol_shape:', x_train.shape[1:])
print('Resized train shape:', x_train.shape)

Resized train vol_shape: (128, 256, 256)
Resized train shape: (400, 128, 256, 256)


In [5]:
import torch

def vxm_data_generator(x_data, batch_size):
    vol_shape = x_data.shape[1:]  # データ形状を取得
    ndims = len(vol_shape)
    
    zero_phi = np.zeros([batch_size, *vol_shape, ndims])
    
    while True:
        idx1 = np.random.randint(0, x_data.shape[0], size=batch_size)
        moving_images = x_data[idx1, ..., np.newaxis]
        # ファインチューニングでは同じ症例同士のペアを避ける
        idx2 = np.random.randint(0, x_data.shape[0], size=batch_size)
        while np.any(idx2 == idx1):
            same_case = idx2 == idx1
            idx2[same_case] = np.random.randint(0, x_data.shape[0], size=same_case.sum())
        fixed_images = x_data[idx2, ..., np.newaxis]

        # TensorFlowからPyTorchのデータ形式に変換
        moving_images = torch.tensor(moving_images).permute(0, 4, 1, 2, 3).float()
        fixed_images = torch.tensor(fixed_images).permute(0, 4, 1, 2, 3).float()

        # チャンネルを最初の次元に追加
        moving_images = moving_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動
        fixed_images = fixed_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動

        inputs = [moving_images, fixed_images]
        outputs = [fixed_images, zero_phi]

        yield (inputs, outputs)

In [6]:
train_generator = vxm_data_generator(x_train, batch_size=2)
in_sample, out_sample = next(train_generator)

# in_sampleとout_sampleの内容を確認する
print("Input Sample Shapes:")
print("Moving Images Shape:", in_sample[0].shape)
print("Fixed Images Shape:", in_sample[1].shape)

print("\nOutput Sample Shapes:")
print("Moved Images (Fixed) Shape:", out_sample[0].shape)
print("Zero Gradient Shape:", out_sample[1].shape)

Input Sample Shapes:
Moving Images Shape: torch.Size([2, 1, 128, 256, 256])
Fixed Images Shape: torch.Size([2, 1, 128, 256, 256])

Output Sample Shapes:
Moved Images (Fixed) Shape: torch.Size([2, 1, 128, 256, 256])
Zero Gradient Shape: (2, 128, 256, 256, 3)


In [7]:
mse_loss = vxm.losses.MSE().loss
grad_loss = vxm.losses.Grad('l2').loss

def total_loss(y_true, y_pred):
    mse = mse_loss(y_true, y_pred)
    grad = grad_loss(y_true, y_pred)
    return mse + 0.01 * grad, mse, grad
#     return mse_loss(y_true, y_pred)

def MSE_Loss(y_true, y_pred):
    y_true = y_true.to(device)
    y_pred = y_pred.to(device)
    mse = mse_loss(y_true, y_pred)
    return mse

def lncc_loss(I, J, window=9, eps=1e-5):
    # I, J: (B, 1, D, H, W)
    padding = window // 2
    weight = torch.ones(1, 1, window, window, window, device=I.device)

    I2 = I * I
    J2 = J * J
    IJ = I * J

    I_sum = F.conv3d(I, weight, padding=padding)
    J_sum = F.conv3d(J, weight, padding=padding)
    I2_sum = F.conv3d(I2, weight, padding=padding)
    J2_sum = F.conv3d(J2, weight, padding=padding)
    IJ_sum = F.conv3d(IJ, weight, padding=padding)

    win_size = window ** 3
    u_I = I_sum / win_size
    u_J = J_sum / win_size

    cross = IJ_sum - u_J * I_sum - u_I * J_sum + u_I * u_J * win_size
    I_var = I2_sum - 2 * u_I * I_sum + u_I * u_I * win_size
    J_var = J2_sum - 2 * u_J * J_sum + u_J * u_J * win_size

    lncc = cross * cross / (I_var * J_var + eps)
    return -torch.mean(lncc)  # maximize LNCC → minimize -LNCC

In [8]:
# configure unet input shape (concatenation of moving and fixed images)
ndim = 3
unet_input_features = 2
# inshape = (*x_train.shape[1:], unet_input_features)

nb_features = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16]
]


In [9]:
import voxelmorph as vxm
import inspect

print(vxm.__file__)
print(vxm.networks.__file__)
print([name for name in dir(vxm.networks) if "VxmDense" in name])

C:\Users\user\anaconda3\envs\nn\lib\site-packages\voxelmorph\__init__.py
C:\Users\user\anaconda3\envs\nn\lib\site-packages\voxelmorph\torch\networks.py
['VxmDense', 'VxmDense1', 'VxmDense2', 'VxmDense_128_256', 'VxmDense_128_256_256']


In [10]:
model3D = vxm.networks.VxmDense_128_256_256((128, 256, 256), nb_features, int_steps=0)
model3D.to(device)
optimizer = optim.Adam(model3D.parameters(), lr=1e-4)

transformer = vxm.layers.SpatialTransformer((64, 128, 128)).to(device)
transformer256 = vxm.layers.SpatialTransformer((128, 256, 256)).to(device)

[64, 128, 128]


C:\Users\user\anaconda3\envs\nn\lib\site-packages\torch\functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3550.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [38]:
import math
from pathlib import Path
import torch
import matplotlib.pyplot as plt

band_names = ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

wavelet_vis_enabled = False
wavelet_vis_every = 100
wavelet_vis_dir = Path('wavelet_stage_outputs')
wavelet_vis_dir.mkdir(exist_ok=True)

class Haar3DAnalysisOnly(nn.Module):
    def __init__(self):
        super().__init__()

        hL = torch.tensor([1.0, 1.0], dtype=torch.float32) / math.sqrt(2.0)
        hH = torch.tensor([1.0, -1.0], dtype=torch.float32) / math.sqrt(2.0)

        filters = []
        names = []

        for z_name, z_filter in zip(['L', 'H'], [hL, hH]):
            for y_name, y_filter in zip(['L', 'H'], [hL, hH]):
                for x_name, x_filter in zip(['L', 'H'], [hL, hH]):
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )
                    filters.append(kernel)
                    names.append(z_name + y_name + x_name)

        weight = torch.stack(filters, dim=0).unsqueeze(1)
        self.register_buffer('weight', weight)
        self.names = names

    def forward(self, x):
        x = F.pad(x, (0, 1, 0, 1, 0, 1))
        return F.conv3d(x, self.weight, stride=1, padding=0)

def analysis_filter_3d(x, analysis_layer):
    return analysis_layer(x)

def down_sampling_3d(w):
    return w[:, :, ::2, ::2, ::2]

def up_sampling_3d(w_down):
    B, C, D, H, W = w_down.shape
    w_up = torch.zeros(
        B, C, D * 2, H * 2, W * 2,
        dtype=w_down.dtype,
        device=w_down.device
    )
    w_up[:, :, ::2, ::2, ::2] = w_down
    return w_up

def make_3d_filter(fz, fy, fx):
    return fz[:, None, None] * fy[None, :, None] * fx[None, None, :]

def create_synthesis_filters(device):
    low = torch.tensor([1.0, 1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)
    high = torch.tensor([1.0, -1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)

    filters = torch.stack([
        make_3d_filter(low, low, low),
        make_3d_filter(low, low, high),
        make_3d_filter(low, high, low),
        make_3d_filter(low, high, high),
        make_3d_filter(high, low, low),
        make_3d_filter(high, low, high),
        make_3d_filter(high, high, low),
        make_3d_filter(high, high, high),
    ], dim=0)

    filters = torch.flip(filters, dims=[1, 2, 3]).unsqueeze(1)
    return filters

def synthesis_filter_3d(w_up, synthesis_filters):
    B, C, D, H, W = w_up.shape
    filtered_bands = []

    for i in range(C):
        band = w_up[:, i:i + 1, :, :, :]
        kernel = synthesis_filters[i:i + 1]
        filtered = F.conv3d(band, kernel, stride=1, padding=1)
        filtered = filtered[:, :, :D, :H, :W]
        filtered_bands.append(filtered)

    filtered_bands = torch.cat(filtered_bands, dim=1)
    reconstructed = torch.sum(filtered_bands, dim=1, keepdim=True)
    return reconstructed, filtered_bands

analysis = Haar3DAnalysisOnly().to(device)
synthesis_filters = create_synthesis_filters(device)
analysis_names = analysis.names

## 80k事前学習モデルのみのWavelet結果


In [ ]:
# Evaluation only: load the 80k pretraining checkpoint; do not fine-tune.
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

PRETRAINED_CHECKPOINT = Path('model_analysis_pipeline_pretrain.pth')
OUTPUT_DIR = Path('pretrain_only_wavelet_evaluation')
OUTPUT_DIR.mkdir(exist_ok=True)

# Fixed different-patient pair for reproducible figures. Change these IDs if needed.
MOVING_PATIENT_ID = 0
FIXED_PATIENT_ID = 1

if not PRETRAINED_CHECKPOINT.exists():
    raise FileNotFoundError(f'80k pretrained checkpoint was not found: {PRETRAINED_CHECKPOINT.resolve()}')
if MOVING_PATIENT_ID == FIXED_PATIENT_ID or max(MOVING_PATIENT_ID, FIXED_PATIENT_ID) >= len(x_train):
    raise ValueError(f'Choose two different valid IDs; x_train contains {len(x_train)} patients.')

try:
    state = torch.load(PRETRAINED_CHECKPOINT, map_location=device, weights_only=True)
except TypeError:
    state = torch.load(PRETRAINED_CHECKPOINT, map_location=device)
model3D.load_state_dict(state['model_state_dict'] if isinstance(state, dict) and 'model_state_dict' in state else state)
model3D.eval()

moving = torch.from_numpy(x_train[MOVING_PATIENT_ID:MOVING_PATIENT_ID + 1]).unsqueeze(1).to(device, dtype=torch.float32)
fixed = torch.from_numpy(x_train[FIXED_PATIENT_ID:FIXED_PATIENT_ID + 1]).unsqueeze(1).to(device, dtype=torch.float32)

with torch.no_grad():
    moving_analysis = analysis_filter_3d(moving, analysis)
    fixed_analysis = analysis_filter_3d(fixed, analysis)
    moving_downsampled = down_sampling_3d(moving_analysis)
    fixed_downsampled = down_sampling_3d(fixed_analysis)
    flow = model3D(moving_downsampled, fixed_downsampled)
    # All bands use the same predicted flow, so this is equivalent to eight separate warps.
    warped_downsampled = transformer(moving_downsampled, flow)
    warped_upsampled = up_sampling_3d(warped_downsampled)
    warped, synthesis_components = synthesis_filter_3d(warped_upsampled, synthesis_filters)

moving_np, fixed_np, warped_np = [item[0, 0].detach().cpu().numpy() for item in (moving, fixed, warped)]
before_error = np.abs(fixed_np - moving_np)
after_error = np.abs(fixed_np - warped_np)
mse_before = (fixed - moving).square().mean().item()
mse_after = (fixed - warped).square().mean().item()

band_labels = globals().get('band_names', ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH'])

def save_eight_bands(tensor, title, filename):
    values = tensor[0].detach().cpu().numpy()
    center = values.shape[1] // 2
    limit = max(np.percentile(np.abs(values[:, center]), 99), 1e-6)
    fig, axes = plt.subplots(2, 4, figsize=(12, 6), constrained_layout=True)
    for index, (ax, label, image) in enumerate(zip(axes.flat, band_labels, values[:, center])):
        if index == 0:
            vmin, vmax = np.percentile(image, [1, 99])
        else:
            vmin, vmax = -limit, limit
        ax.imshow(image, cmap='gray', vmin=vmin, vmax=vmax)
        ax.set_title(label)
        ax.axis('off')
    fig.suptitle(title, fontsize=14)
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved: {path.resolve()}')

save_eight_bands(moving_analysis, '1. Moving: Haar analysis filter output (8 bands)', '01_moving_analysis.png')
save_eight_bands(moving_downsampled, '2. Moving: downsampled Wavelet bands', '02_moving_downsampled.png')
save_eight_bands(warped_upsampled, '3. Moving: warped and upsampled Wavelet bands (80k pretraining only)', '03_warped_upsampled.png')
save_eight_bands(synthesis_components, '4. Moving: synthesis-filter components', '04_synthesis_components.png')

image_vmin, image_vmax = np.percentile(np.concatenate([moving_np.ravel(), fixed_np.ravel(), warped_np.ravel()]), [1, 99])
error_vmax = max(np.percentile(np.concatenate([before_error.ravel(), after_error.ravel()]), 99), 1e-6)
views = [
    ('Axial', lambda a: a[a.shape[0] // 2]),
    ('Coronal', lambda a: a[:, a.shape[1] // 2, :]),
    ('Sagittal', lambda a: a[:, :, a.shape[2] // 2]),
]
fig, axes = plt.subplots(3, 5, figsize=(18, 11), constrained_layout=True)
for row, (orientation, slicer) in enumerate(views):
    panels = [
        ('Moving', slicer(moving_np), 'gray', image_vmin, image_vmax),
        ('Fixed', slicer(fixed_np), 'gray', image_vmin, image_vmax),
        ('Warped (80k pretraining)', slicer(warped_np), 'gray', image_vmin, image_vmax),
        ('|Fixed − moving|', slicer(before_error), 'magma', 0, error_vmax),
        ('|Fixed − warped|', slicer(after_error), 'magma', 0, error_vmax),
    ]
    for col, (label, image, cmap, vmin, vmax) in enumerate(panels):
        ax = axes[row, col]
        im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(f'{orientation}: {label}')
        ax.axis('off')
        if col == 4:
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle(
    f'Pretraining-only registration (patient {MOVING_PATIENT_ID} → {FIXED_PATIENT_ID})\n'
    f'MSE: {mse_before:.5f} → {mse_after:.5f} | no fine-tuning performed', fontsize=15
)
result_path = OUTPUT_DIR / '05_pretraining_only_registration.png'
fig.savefig(result_path, dpi=200, bbox_inches='tight')
plt.show()

print(f'Loaded 80k pretraining checkpoint: {PRETRAINED_CHECKPOINT.resolve()}')
print(f'Input / analysis / downsampled / upsampled / synthesis / warped shapes: {tuple(moving.shape)} / {tuple(moving_analysis.shape)} / {tuple(moving_downsampled.shape)} / {tuple(warped_upsampled.shape)} / {tuple(synthesis_components.shape)} / {tuple(warped.shape)}')
print(f'MSE: before={mse_before:.6f}, after={mse_after:.6f}')
print(f'Saved files: {OUTPUT_DIR.resolve()}')
